# Phase 4: Station reliability

In this notebook we count how many train disruptions each station had in 2021–2023. We use this as a measure of how reliable a station is. The result is one row per station, which we later link to the neighbourhoods (Phase 5).


In [1]:
import os                  # for file paths
import pandas as pd        # for tables

# Folder with the clean data from Phase 1 and 2
clean_folder = 'data/cleaned_data'

# Load the cleaned data from Phase 1
storingen = pd.read_pickle(os.path.join(clean_folder, 'storingen.pkl'))
stations = pd.read_pickle(os.path.join(clean_folder, 'stations.pkl'))

print(len(storingen))


15541


In [2]:
# Remove disruptions without a station code
storingen = storingen[storingen['rdt_station_codes'].notna()]

# Remove planned engineering work
storingen = storingen[storingen['cause_group'] != 'engineering work']

print(len(storingen))

storingen.head()

14928


,rdt_id,ns_lines,rdt_lines,rdt_lines_id,rdt_station_names,rdt_station_codes,cause_nl,cause_en,statistical_cause_nl,statistical_cause_en,cause_group,start_time,end_time,duration_minutes
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0
1,35627,Amsterdam-Almere; Amsterdam-Amersfoort,"Amersfoort - Amsterdam Centraal, Amsterdam Cen...","135,145","Amsterdam Centraal,Amsterdam Muiderpoort,Amste...","ASD, ASDM, ASSP, DMN, WP",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 10:17:19,2021-01-01 10:29:38,12.0
2,35628,Utrecht-Baarn,Baarn - Utrecht Centraal,44,"Baarn,Den Dolder,Soestdijk,Soest,Soest Zuid","BRN, DLD, SD, ST, STZ",defecte bovenleiding,damaged overhead wires,defecte bovenleiding,damaged overhead wires,infrastructure,2021-01-01 10:44:22,2021-01-01 17:44:52,421.0
3,35629,Zwolle-Emmen,Emmen - Zwolle,159,"Gramsbergen,Hardenberg,Mariënberg","GBG, HDB, MRB",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 12:25:19,2021-01-01 12:45:00,20.0
4,35630,Roosendaal-Vlissingen,Roosendaal - Vlissingen,21,"Bergen op Zoom,Rilland-Bath","BGN, RB",aanrijding,collision,aanrijding,collision,accidents,2021-01-01 15:17:02,2021-01-01 15:58:59,42.0


In [3]:
# Split the text "HGL, HGLO, ODZ" into a list ["HGL", " HGLO", " ODZ"] because we want to give every station in the list its own row
storingen['station_code'] = storingen['rdt_station_codes'].str.split(',')

# Give every station in the list its own row so that we can join the disruptions to the stations
storingen = storingen.explode('station_code')

# Remove the spaces around the codes (" HGLO" becomes "HGLO") otherwise the join with the stations will not work
storingen['station_code'] = storingen['station_code'].str.strip()

print(len(storingen))

# Check in the first few rows that every row now has one station
storingen.head()

67850


,rdt_id,ns_lines,rdt_lines,rdt_lines_id,rdt_station_names,rdt_station_codes,cause_nl,cause_en,statistical_cause_nl,statistical_cause_en,cause_group,start_time,end_time,duration_minutes,station_code
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,HGL
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,HGLO
0,35626,Hengelo-Bielefeld,"Berlin Ostbahnhof - Hengelo, Bielefeld Hbf - H...","91,124,174","Hengelo,Hengelo Oost,Oldenzaal","HGL, HGLO, ODZ",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 08:12:55,2021-01-01 11:30:00,197.0,ODZ
1,35627,Amsterdam-Almere; Amsterdam-Amersfoort,"Amersfoort - Amsterdam Centraal, Amsterdam Cen...","135,145","Amsterdam Centraal,Amsterdam Muiderpoort,Amste...","ASD, ASDM, ASSP, DMN, WP",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 10:17:19,2021-01-01 10:29:38,12.0,ASD
1,35627,Amsterdam-Almere; Amsterdam-Amersfoort,"Amersfoort - Amsterdam Centraal, Amsterdam Cen...","135,145","Amsterdam Centraal,Amsterdam Muiderpoort,Amste...","ASD, ASDM, ASSP, DMN, WP",defecte trein,broken down train,defecte trein,broken down train,rolling stock,2021-01-01 10:17:19,2021-01-01 10:29:38,12.0,ASDM


In [4]:
# Keep only station codes that are in the Dutch station list
storingen = storingen[storingen['station_code'].isin(stations['code'])]

print(len(storingen))

65896


In [5]:
#calculation For each station we calculate:the average number of disruptions per year,the average number of disruption minutes per year
#We divide by 3 because the data covers three years (2021, 2022 and 2023).

# Count the number of disruptions per station
aantal = storingen.groupby('station_code')['rdt_id'].count()

# Add up the disruption minutes per station
minuten = storingen.groupby('station_code')['duration_minutes'].sum()

# Put both in one table and divide by 3 to get the average per year
per_station = pd.DataFrame({
    'storingen_per_jaar': aantal / 3,
    'minuten_per_jaar': minuten / 3,
})
per_station = per_station.reset_index()

per_station.head()

,station_code,storingen_per_jaar,minuten_per_jaar
0,AC,76.000000,17321.333333
1,AH,251.666667,49488.666667
2,AHP,120.666667,26725.333333
3,AHPR,37.666667,2675.333333
4,AHZ,97.000000,16672.000000


In [6]:
# Start from the full list of stations
alle_stations = stations[['code']].rename(columns={'code': 'station_code'})

# Add the disruption numbers
storingen_per_station = alle_stations.merge(per_station, on='station_code', how='left')

# Stations without disruptions get 0 instead of an empty value
storingen_per_station = storingen_per_station.fillna(0)

print(len(storingen_per_station))
storingen_per_station.sort_values('storingen_per_jaar', ascending=False).head(10)

397


,station_code,storingen_per_jaar,minuten_per_jaar
288,RTD,535.333333,82828.000000
303,SHL,433.333333,81266.666667
330,UT,355.666667,62043.666667
22,ASD,311.000000,70137.333333
68,BD,299.333333,27403.333333
37,AH,251.666667,49488.666667
28,ASS,251.000000,46784.666667
17,AMF,241.000000,48713.666667
226,LEDN,239.000000,34555.333333
111,DDR,215.666667,32887.333333


## Share of disrupted trains per station (2023)


In [7]:
# Feedback: a big station has many disruptions simply because many trains stop there.
# So instead of counting disruptions, we look at the share (%) of all trains at a station that were disrupted.
# For this we use the train archive of Rijden de Treinen: every row is one train stopping at one station.
# Download the 12 monthly files of 2023 (services-2023-01 up to services-2023-12) from
# https://www.rijdendetreinen.nl/en/open-data/train-archive and put them (unzipped, as .csv) in the folder data/raw/treinarchief/

treinarchief_folder = 'data/raw/treinarchief'    # separate folder, so data/raw stays tidy

# The month numbers as they are written in the file names
maand_nummers = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

maanden = []
for maand in maand_nummers:
    bestand = os.path.join(treinarchief_folder, 'services-2023-' + maand + '.csv')    # for example services-2023-01.csv
    ritten = pd.read_csv(bestand)                                    # one month of train stops
    ritten['Stop:Station code'] = ritten['Stop:Station code'].fillna('NA')   # 'NA' (Nieuw Amsterdam) is read as empty, see Phase 1

    # A stop counts as cancelled if the arrival or the departure at that station was cancelled
    uitgevallen = (ritten['Stop:Arrival cancelled'] == True) | (ritten['Stop:Departure cancelled'] == True)

    # A stop counts as delayed if the train arrived or left 5 minutes or more late
    vertraagd = (ritten['Stop:Arrival delay'] >= 5) | (ritten['Stop:Departure delay'] >= 5)

    # Disrupted = cancelled or delayed (True/False for every stop)
    ritten['verstoord'] = uitgevallen | vertraagd

    # Per station: count all trains, and add up the disrupted ones (True counts as 1)
    treinen = ritten.groupby('Stop:Station code')['verstoord'].count()
    verstoord = ritten.groupby('Stop:Station code')['verstoord'].sum()

    per_maand = pd.DataFrame({'treinen': treinen, 'verstoord': verstoord}).reset_index()
    maanden.append(per_maand)
    print('month', maand, ':', len(ritten), 'stops')


month 01 : 1742944 stops
month 02 : 1606345 stops
month 03 : 1792961 stops
month 04 : 1739663 stops
month 05 : 1767831 stops
month 06 : 1787886 stops
month 07 : 1811194 stops
month 08 : 1802990 stops
month 09 : 1757341 stops
month 10 : 1826745 stops
month 11 : 1794007 stops
month 12 : 1809486 stops


In [8]:
# Put the 12 months under each other and add them up per station
treinen_per_station = pd.concat(maanden)
treinen_per_station = treinen_per_station.groupby('Stop:Station code')[['treinen', 'verstoord']].sum()
treinen_per_station = treinen_per_station.reset_index()
treinen_per_station = treinen_per_station.rename(columns={'Stop:Station code': 'station_code'})

# Share of disrupted trains in % and the average number of trains per day
treinen_per_station['pct_verstoord'] = treinen_per_station['verstoord'] / treinen_per_station['treinen'] * 100
treinen_per_station['treinen_per_dag'] = treinen_per_station['treinen'] / 365

treinen_per_station.sort_values('treinen', ascending=False).head(10)


,station_code,treinen,verstoord,pct_verstoord,treinen_per_dag
478,UT,431204,45404,10.529587,1181.380822
35,ASD,393750,61462,15.609397,1078.767123
449,SHL,276521,46235,16.720249,757.591781
42,ASS,273650,37239,13.608259,749.726027
546,ZL,271564,32765,12.065296,744.010959
434,RTD,265645,52009,19.578385,727.794521
2,AH,227062,32208,14.184672,622.087671
229,GVC,219758,24215,11.018939,602.076712
17,AMF,211265,22989,10.881594,578.808219
332,LEDN,210425,31840,15.131282,576.506849


In [9]:
# Add the new columns to the station table we already made
# how='left' keeps all Dutch stations; foreign stations from the archive are left out
storingen_per_station = storingen_per_station.merge(
    treinen_per_station[['station_code', 'pct_verstoord', 'treinen_per_dag']],
    on='station_code', how='left',
)

# Check: how many stations did not get a value (isna() is True for an empty cell, sum() counts the Trues)
print('Stations without train data:', storingen_per_station['pct_verstoord'].isna().sum())
storingen_per_station.sort_values('pct_verstoord', ascending=False).head(10)


Stations without train data: 0


,station_code,storingen_per_jaar,minuten_per_jaar,pct_verstoord,treinen_per_dag
291,RTST,89.000000,12064.000000,36.445783,19.101370
137,EGHM,42.000000,32997.333333,31.475321,40.353425
147,GS,40.000000,12574.000000,26.709554,126.484932
53,BGN,40.333333,12989.000000,26.156385,117.868493
219,KRG,41.333333,13452.666667,24.106432,86.079452
209,BZL,32.666667,11656.000000,23.942332,87.035616
281,RB,35.333333,12274.000000,23.612545,84.909589
216,KBD,33.000000,12127.666667,23.170771,84.884932
244,MDB,31.666667,11671.000000,22.999902,111.912329
68,BD,299.333333,27403.333333,22.148928,270.794521


In [10]:
storingen_per_station.to_pickle(os.path.join(clean_folder, 'storingen_per_station.pkl'))